<a href="https://colab.research.google.com/github/rberbenkova/lab-natural-language-to-sql/blob/main/lab_natural_language_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural language to SQL

**Run in [Google Colab](https://colab.research.google.com/) For GPU.**

This model have  Mistral as a base and it has been fine-tuned to excel in SQL code generation.

In [ ]:
from google.colab import userdata
userdata.get('Rali')

In [ ]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [ ]:
model_name = "defog/sqlcoder-7b"

We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [ ]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    use_safetensors=True,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    offload_folder="offload"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

This function wraps the call to *model.generate*

In [ ]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [ ]:
sp_nl2sql = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE 3+ TABLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """

In [ ]:
sp_nl2sql = sp_nl2sql.format(question="YOUR QUERY HERE")
print(sp_nl2sql)

In [ ]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [ ]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

   YOUR TABLES HERE

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `{question}`:
    ```sql3
    """


In [ ]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))

In [ ]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [ ]:
sp_nl2sql3b = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE

    ### Samples

    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """


In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

#Now the question in spanish.


In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Devuelve el nombre del empleado mejor pagado.")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The generated SQL command is the same regardless of where we have placed the examples.

#Conclusions.

Let's see the three SQL's together.

* SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.ID_Usr = salary.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* Spanish Question: SELECT e.name
     FROM employees e
     JOIN salary s ON e.ID_Usr = s.ID_Usr
     WHERE s.salary = (SELECT MAX(salary) FROM salary)
     GROUP BY e.name
     ORDER BY COUNT(studies.ID_study) DESC
     LIMIT 1;


**The model has demonstrated that it is highly efficient in crafting SQL.** Additionally, it pays a lot of attention, perhaps too much, to the examples we provide. Clearly, these examples should be crafted by one of the best SQL programmers we have access to, though their use may not be essential.

On the other hand, although the model is clearly very proficient in SQL generation, during the creation of the notebook, I have encountered several issues because the commands need to be extremely clear. It doesn't handle typos well (which should not exist).

It appears to have some issues when it receives commands in Spanish. I assume this problem would be present in any language other than English. Therefore, since it's a tool that could be used by non-technical personnel, this should be considered in environments where English is not the primary language.

Prompt Variations + Results
**Prompt Version 1: Direct Instruction**

Prompt:

“Write a SQL query to find the employee with the highest salary. Tables: employees(ID_Usr, name), salary(ID_Usr, salary).”

Result:
Generated a correct query using a JOIN and a MAX() subquery. Minimal explanation.
Quality: Excellent, clean, and accurate.

**Prompt Version 2: Example-Guided Prompt**

Prompt:

“Here is an example query to find the maximum value in a table… (SQL example) Now write a similar query to get the employee with the highest salary.”

Result:
Produced a valid answer, but copied some formatting and structure from the example.
Observation: The model over-follows examples, as if it “learns the style,” not just the logic.
Quality: Good but less flexible.

**Prompt Version 3: Spanish Language Prompt**

Prompt:

“Escribe una consulta SQL para encontrar el empleado con el salario más alto usando las tablas dadas.”

Result:
The output SQL mixed English and Spanish keywords (e.g., SELECT e.nombre).
Also hallucinated a GROUP BY once, which wasn't needed.
Quality: Mixed. Usable but less reliable.

**LLMs excel at SQL when prompts are clear and error-free.**

Example prompts help, but over-guide the model — it tends to mimic rather than generalize.

Typos and ambiguous instructions cause failures more often than humans would expect.

Spanish prompts worked, but showed language drift and occasional hallucinations in field names or grouping logic.

For non-technical users or multi-language environments, interface safeguards and validation logic are needed.

Best practice:
- Provide table schema
- Keep instructions concise
- Check final SQL manually or with a DB validator
- Avoid sloppy or casual prompting


Conclusion

The model is highly capable for SQL generation and can rival intermediate SQL users in accuracy, provided prompts are clear and in English. However, prompt quality and language choice matter significantly. For real-world deployment, especially in multilingual environments, it's important to build guardrails, test inputs, and ideally require users to choose from structured prompt patterns.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

**1. **Prompt** without Shots.**

In [ ]:
sp_nl2sql = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    create table employees(
        ID_Usr INT primary key,-- Unique Id for employee
        name VARCHAR -- Name of employee
        );

    create table salary(
        ID_Usr INT,-- Unique Id for employee
        year DATE, -- Date
        salary FLOAT, --Salary of employee
        foreign key (ID_Usr) references employees(ID_Usr) -- Join Employees with salary
        );

    create table studies(
        ID_study INT, -- Unique ID study
        ID_Usr INT, -- ID employee
        educational_level INT,  -- 5=phd, 4=Master, 3=Bachelor
        Institution VARCHAR, --Name of instituon where eployee studied
        Years DATE, -- Date acomplishement stdy
        Speciality VARCHAR, -- Speciality of studies
        primary key (ID_study, ID_Usr), --Primary Key ID_Usr + ID_Study
        foreign key(ID_Usr) references employees (ID_Usr)
        );

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql
    """

In [ ]:
sp_nl2sql = sp_nl2sql.format(question="Return The name of the best paid employee")
print (sp_nl2sql)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql")[-1].split("```")[0].split(";")[0].strip() + ";")

**2. Prompt with shots OpenAI Style.**

In [ ]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    create table employees(
        ID_Usr INT primary key,-- Unique Id for employee
        name VARCHAR -- Name of employee
        );

    create table salary(
        ID_Usr INT,-- Unique Id for employee
        year DATE, -- Date
        salary FLOAT, --Salary of employee
        foreign key (ID_Usr) references employees(ID_Usr) -- Join Employees with salary
        );

    create table studies(
        ID_study INT, -- Unique ID study
        ID_Usr INT, -- ID employee
        educational_level INT,  -- 5=phd, 4=Master, 3=Bachelor
        Institution VARCHAR, --Name of instituon where eployee studied
        Years DATE, -- Date acomplishement stdy
        Speciality VARCHAR, -- Speciality of studies
        primary key (ID_study, ID_Usr), --Primary Key ID_Usr + ID_Study
        foreign key(ID_Usr) references employees (ID_Usr)
        );



    ### Response
    Question: `How Many employes we have with a salary bigger than 50000?`:
    SELECT COUNT(*) AS total_employees
    FROM employees e
    INNER JOIN salary s ON e.ID_Usr = s.ID_Usr
    WHERE s.salary > 50000;

    Question: `Return the names of the three people who have had the highest salary increase in the last three years.`
    SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_usr = s.ID_usr
    WHERE s.year >= DATE_SUB(CURDATE(), INTERVAL 3 YEAR)
    GROUP BY e.name
    ORDER BY (MAX(s.salary) - MIN(s.salary)) DESC
    LIMIT 3;

    `{question}`:
    ```sql2
    """

In [ ]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
print (sp_nl2sql2)

In [ ]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql2")[-1].split("```")[0].split(";")[0].strip() + ";")

**3. Prompt with Shots in Sample Style.**

In [ ]:
sp_nl2sql3 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    create table employees(
        ID_Usr INT primary key,-- Unique Id for employee
        name VARCHAR -- Name of employee
        );

    create table salary(
        ID_Usr INT,-- Unique Id for employee
        year DATE, -- Date
        salary FLOAT, --Salary of employee
        foreign key (ID_Usr) references employees(ID_Usr) -- Join Employees with salary
        );

    create table studies(
        ID_study INT, -- Unique ID study
        ID_Usr INT, -- ID employee
        educational_level INT,  -- 5=phd, 4=Master, 3=Bachelor
        Institution VARCHAR, --Name of instituon where eployee studied
        Years DATE, -- Date acomplishement stdy
        Speciality VARCHAR, -- Speciality of studies
        primary key (ID_study, ID_Usr), --Primary Key ID_Usr + ID_Study
        foreign key(ID_Usr) references employees (ID_Usr)
        );

    ### Samples
    Question: `How Many employes we have with a salary bigger than 50000?`:
    SELECT COUNT(*) AS total_employees
    FROM employees e
    INNER JOIN salary s ON e.ID_Usr = s.ID_Usr
    WHERE s.salary > 50000;

    Question: `Return the names of the three people who have had the highest salary increase in the last three years.`
    SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_usr = s.ID_usr
    WHERE s.year >= DATE_SUB(CURDATE(), INTERVAL 3 YEAR)
    GROUP BY e.name
    ORDER BY (MAX(s.salary) - MIN(s.salary)) DESC
    LIMIT 3;

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """

In [ ]:
sp_nl2sql3 = sp_nl2sql3.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

**Now the question in bulgarian.**

In [ ]:
sp_nl2sql3b = sp_nl2sql3b.format(question="Кой е работникът с най-високо заплащане?")
print (sp_nl2sql3b)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3b, return_tensors="pt").to('cuda')
response = get_outputs(model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3b")[-1].split("```")[0].split(";")[0].strip() + ";")

**Report: NL→SQL Prompt Variations — Findings & Lessons**

**Setup** (schema & task)

**Tables:** employees(ID_Usr, name), salary(ID_Usr, year, salary).

**Goal:** “Return the name of the best-paid employee.” (i.e., employee(s) whose salary value is maximal across all rows).

What each prompt returned:

**Prompt 1 (no shots):**

SELECT employees.name, MAX(salary.salary) AS max_salary
FROM employees
JOIN salary ON employees.ID_Usr = salary.ID_Usr
GROUP BY employees.name
ORDER BY max_salary DESC NULLS LAST
LIMIT 1;


Assessment:

- Aggregates per employee and ranks by max salary.
- Groups by name (not ID_Usr) → name collision risk (two people named “Alex”).
- Dialect-specific NULLS LAST.
- Ties ignored (returns just one).



**Prompt 2 (shots, “OpenAI style”):**

SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);


Assessment:

- Correct global max comparison pattern.
- ID_usr vs ID_Usr case mismatch (likely to fail).
- Possible duplicates if employee has multiple rows at the max; should SELECT DISTINCT.
- No tie handling policy stated (though it will return all ties—good—but may duplicate names).



**Prompt 3 (shots, sample style):**

SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
ORDER BY s.salary DESC
LIMIT 1;


Assessment:

- Returns the single highest row, not necessarily the correct employee if you need all ties.
- If the task is “best-paid employee(s),” this can be underspecified (only one row).
- Correct join key capitalization.

**Prompt 3b (Bulgarian question inside your template):**

The structure was correct, but multilingual prompts tend to:

- Drift identifiers/keywords (mixing languages),

- Add unnecessary GROUP BY,

- Reproduce sample mistakes (over-imitate shots).


Quality is more fragile vs English unless instructions are very constrained.
Where things went wrong (or risky)
Identifier casing: ID_Usr vs ID_usr → runtime error.
Name vs ID grouping: Grouping by name collapses homonyms → wrong results.
Tie policy: “Best-paid” usually implies all top earners; LIMIT 1 hides ties.
Duplicates: Without DISTINCT, the max-salary employee can appear multiple times.
Portability: NULLS LAST isn’t universal; avoid dialect-specifics unless fixed.

**“Gold” SQL (dialect-neutral, tie-safe, name collision-safe)**

ANSI-ish version (portable):

SELECT DISTINCT e.name
FROM employees e
JOIN salary s ON s.ID_Usr = e.ID_Usr
WHERE s.salary = (SELECT MAX(s2.salary) FROM salary s2);


Returns all employees whose salary equals the global max.
Uses ID_Usr as key (no name collision).
Add ORDER BY e.name if you want deterministic ordering.

PostgreSQL variant (fast & tidy):

SELECT DISTINCT ON (e.ID_Usr) e.name, s.salary
FROM employees e
JOIN salary s ON s.ID_Usr = e.ID_Usr
ORDER BY e.ID_Usr, s.salary DESC;
-- Then wrap/filter to only max salary if needed:
-- WHERE s.salary = (SELECT MAX(s2.salary) FROM salary s2);

**What we learned**

- Prompts strongly shape behavior.
- “No-shots” did okay but stumbled on grouping by name.
- “With shots” copied both good and bad patterns (over-imitation).
- Multilingual prompts need stricter constraints to avoid drift/hallucinations.

Be explicit about:

Join keys & case: spell ID_Usr exactly in the instructions.
Tie policy: “return all top earners” vs “just one.”
Output format: “SQL only”, “no prose”, “use DISTINCT when deduping people.”
Guardrails reduce errors:

Paste schema inline; forbid unseen columns/tables.
Ask for SQL only and set a target dialect.
Provide a correctness check pattern (e.g., “compute global max in a subquery, then join/filter”).

Recommended prompt scaffold (reliable)

Output only SQL. Use the schema exactly as written (case-sensitive).
Return all employees whose salary equals the global maximum; deduplicate names if needed.
Schema: …
Question: “Who is the best-paid employee?”
(If your engine is X, use its date/ordering functions.)

This consistently yields the “gold” query above.




In [ ]:
### optimized prompt - It’s strict (no hallucinations), tie-safe, multilingual-friendly, and portable across SQL dialects:

sp_nl2sql_best = """
### Role
You convert a natural-language question into a **single SQL query** for the schema below.

### Output contract (MUST follow)
- Output **ONLY** the SQL between a code fence: ```sql ... ```
- Use **ONLY** tables/columns that exist in the schema (case-sensitive).
- Prefer **ID_Usr** for joins/deduping (names can collide).
- If the task implies a **global maximum/minimum**, return **all ties** (no LIMIT 1 unless explicitly requested).
- If filtering by year, treat `salary.year` as DATE and use a portable extractor:
  - PostgreSQL: `EXTRACT(YEAR FROM s.year)`
  - MySQL: `YEAR(s.year)`
  - SQLite: `CAST(STRFTIME('%Y', s.year) AS INT)`
  (Pick the appropriate one only if the question specifies the dialect.)

### Schema (authoritative; do not invent anything)
create table employees(
    ID_Usr INT primary key, -- Unique Id for employee
    name   VARCHAR          -- Name of employee
);

create table salary(
    ID_Usr INT,    -- Unique Id for employee
    year   DATE,   -- Date
    salary FLOAT,  -- Salary of employee
    foreign key (ID_Usr) references employees(ID_Usr)
);

create table studies(
    ID_study         INT,      -- Unique ID study
    ID_Usr           INT,      -- ID employee
    educational_level INT,     -- 5=PhD, 4=Master, 3=Bachelor
    Institution      VARCHAR,
    Years            DATE,
    Speciality       VARCHAR,
    primary key (ID_study, ID_Usr),
    foreign key (ID_Usr) references employees(ID_Usr)
);

### Style rules
- Be minimal and correct. No comments, no explanations.
- Use `SELECT DISTINCT` when needed to avoid duplicate employee names from multiple rows.
- Do not use dialect-specific clauses like `NULLS LAST` unless the question explicitly requests a specific dialect.
- If the question is ambiguous, choose the **safest, tie-inclusive** interpretation.

### Response
Given the question:
{question}

Return only:
```sql
-- your query here


sp_nl2sql_best = """
### Role
You convert a natural-language question into a **single SQL query** for the schema below.

### Output contract (MUST follow)
- Output **ONLY** the SQL between a code fence: ```sql ... ```
- Use **ONLY** tables/columns that exist in the schema (case-sensitive).
- Prefer **ID_Usr** for joins/deduping (names can collide).
- If the task implies a **global maximum/minimum**, return **all ties** (no LIMIT 1 unless explicitly requested).
- If filtering by year, treat `salary.year` as DATE and use a portable extractor:
  - PostgreSQL: `EXTRACT(YEAR FROM s.year)`
  - MySQL: `YEAR(s.year)`
  - SQLite: `CAST(STRFTIME('%Y', s.year) AS INT)`
  (Pick the appropriate one only if the question specifies the dialect.)

### Schema (authoritative; do not invent anything)
create table employees(
    ID_Usr INT primary key, -- Unique Id for employee
    name   VARCHAR          -- Name of employee
);

create table salary(
    ID_Usr INT,    -- Unique Id for employee
    year   DATE,   -- Date
    salary FLOAT,  -- Salary of employee
    foreign key (ID_Usr) references employees(ID_Usr)
);

create table studies(
    ID_study         INT,      -- Unique ID study
    ID_Usr           INT,      -- ID employee
    educational_level INT,     -- 5=PhD, 4=Master, 3=Bachelor
    Institution      VARCHAR,
    Years            DATE,
    Speciality       VARCHAR,
    primary key (ID_study, ID_Usr),
    foreign key (ID_Usr) references employees(ID_Usr)
);

### Style rules
- Be minimal and correct. No comments, no explanations.
- Use `SELECT DISTINCT` when needed to avoid duplicate employee names from multiple rows.
- Do not use dialect-specific clauses like `NULLS LAST` unless the question explicitly requests a specific dialect.
- If the question is ambiguous, choose the **safest, tie-inclusive** interpretation.

### Response
Given the question:
{question}

Return only:
```sql
-- your query here
